# APIC_A Reproduction Notebook

Notebook ini mereproduksi hasil:
- Konversi weight dan bias dari checkpoint ke .memh/.vh (int8).
- Paket RTL lengkap dengan SRAM dari folder APIC_A.
- Arsip output siap dibagikan.

Jalankan sel dari atas ke bawah.

In [8]:
from pathlib import Path

ROOT = Path.cwd()
APIC_DIR = ROOT 
WEIGHTS_DIR = APIC_DIR / "weights" / "mnist_gan_mlp"
OUT_WEIGHTS_DIR = APIC_DIR / "weights_vh" / "mnist_gan_mlp"
RTL_DIR = APIC_DIR / "rtl"
ARTIFACTS_DIR = APIC_DIR / "artifacts"
EXPORT_DIR = ARTIFACTS_DIR / "export_apic_a"

for path in [APIC_DIR, WEIGHTS_DIR, RTL_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path: {path}")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Checkpoints:", WEIGHTS_DIR)
print("Weights output:", OUT_WEIGHTS_DIR)
print("RTL source:", RTL_DIR)
print("Export dir:", EXPORT_DIR)

Project root: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A
Checkpoints: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights\mnist_gan_mlp
Weights output: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights_vh\mnist_gan_mlp
RTL source: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\rtl
Export dir: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\artifacts\export_apic_a


## Dependencies

Notebook ini butuh Python 3 dan torch (CPU cukup).
Jika torch belum terpasang, install dahulu lewat pip.

In [9]:
import importlib.util
import sys

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Missing dependency: torch. Install with: pip install torch")

import torch

print("python:", sys.version)
print("torch:", torch.__version__)

python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
torch: 2.12.0+cpu


## Konversi weight dan bias

Script akan mengonversi checkpoint GAN MNIST ke int8 .memh/.vh.
Nama file mengikuti prefix D300_ dan G300_.

In [10]:
import subprocess
import sys

script = APIC_DIR / "scripts" / "ckpt_to_vh.py"
if not script.exists():
    raise FileNotFoundError(f"Missing script: {script}")

OUT_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

def run_convert(ckpt_name: str, prefix: str) -> None:
    ckpt_path = WEIGHTS_DIR / ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")
    cmd = [
        sys.executable,
        str(script),
        "--ckpt", str(ckpt_path),
        "--out", str(OUT_WEIGHTS_DIR),
        "--prefix", prefix,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

run_convert("D--300.ckpt", "D300_")
run_convert("G--300.ckpt", "G300_")

print("Conversion complete.")

Running: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\.venv\Scripts\python.exe d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\scripts\ckpt_to_vh.py --ckpt d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights\mnist_gan_mlp\D--300.ckpt --out d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights_vh\mnist_gan_mlp --prefix D300_
Running: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\.venv\Scripts\python.exe d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\scripts\ckpt_to_vh.py --ckpt d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights\mnist_gan_mlp\G--300.ckpt --out d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights_vh\mnist_gan_mlp --prefix G300_
Conversion complete.


## Verifikasi output weight

Ringkas isi weights_manifest dan daftar file .memh/.vh yang dibuat.

In [11]:
import json

manifest_path = OUT_WEIGHTS_DIR / "weights_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Missing manifest: {manifest_path}")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
tensors = manifest.get("tensors", {})
print("Sources:")
for src in manifest.get("sources", []):
    print(" -", src)

print(f"Tensor count: {len(tensors)}")
for name in sorted(tensors):
    entry = tensors[name]
    print(f"{name}: shape={entry['shape']} scale={entry['scale']}")

print("\nGenerated .memh:")
for path in sorted(OUT_WEIGHTS_DIR.glob("*.memh")):
    print(" -", path.name)

print("Generated .vh:")
for path in sorted(OUT_WEIGHTS_DIR.glob("*.vh")):
    print(" -", path.name)

Sources:
 - APIC_A\weights\mnist_gan_mlp\D--300.ckpt
 - APIC_A\weights\mnist_gan_mlp\G--300.ckpt
 - d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights\mnist_gan_mlp\D--300.ckpt
 - d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\weights\mnist_gan_mlp\G--300.ckpt
Tensor count: 12
D300_0_bias: shape=[256] scale=0.011188526791850412
D300_0_weight: shape=[256, 784] scale=0.011932632115882212
D300_2_bias: shape=[256] scale=0.012826556295860471
D300_2_weight: shape=[256, 256] scale=0.006587257066110927
D300_4_bias: shape=[1] scale=0.012192740215091254
D300_4_weight: shape=[1, 256] scale=0.0024597482887778696
G300_0_bias: shape=[256] scale=0.011787274691063589
G300_0_weight: shape=[256, 64] scale=0.008383775320578747
G300_2_bias: shape=[256] scale=0.007673112426217147
G300_2_weight: shape=[256, 256] scale=0.028844245775478092
G300_4_bias: shape=[784] scale=0.006360630350788747
G300_4_weight: shape=[784, 256] scale=0.03370054500309501

Generated .memh:
 - D300_0_bias.memh
 - D300_0

## Paket RTL lengkap dengan SRAM

Salin seluruh RTL APIC_A (termasuk subfolder SRAM) ke folder ekspor.

In [12]:
import shutil

rtl_out = EXPORT_DIR / "rtl"
if rtl_out.exists():
    shutil.rmtree(rtl_out)

shutil.copytree(RTL_DIR, rtl_out)

rtl_files = list(rtl_out.rglob("*.v"))
print("RTL files copied:", len(rtl_files))
print("RTL output:", rtl_out)

RTL files copied: 10
RTL output: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\artifacts\export_apic_a\rtl


## Buat arsip output

Gabungkan RTL dan hasil weights ke satu arsip zip.

In [13]:
import shutil
import zipfile

weights_out = EXPORT_DIR / "weights_vh"
if weights_out.exists():
    shutil.rmtree(weights_out)

shutil.copytree(OUT_WEIGHTS_DIR, weights_out)

zip_path = ARTIFACTS_DIR / "export_apic_a.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(EXPORT_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(EXPORT_DIR))

print("Export folder:", EXPORT_DIR)
print("Zip archive:", zip_path)

Export folder: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\artifacts\export_apic_a
Zip archive: d:\SEMESTER8\Volume-IIC-OSIC\APIC_A_DONE (1)\APIC_A\artifacts\export_apic_a.zip
